# Module 16 — Introduction to Computational Thinking

**Course:** Programming Fundamentals Using Python

**Prerequisites:** Modules 7–15. This module steps back from *syntax* to
*how to think*: the habits that make programs correct, readable, fast
enough, and easy to change. It also contains the **final integrated
project**, which combines everything from Modules 7–16.

**What you will learn:**

- Abstraction, decomposition and generalisation — with code
- Algorithmic thinking: comparing two solutions to the same problem
- Efficiency and the idea of growth: O(1), O(n), O(n²)
- Choosing the right data structure (list vs tuple vs set vs dict)
- Writing maintainable code: names, functions, docstrings, organisation
- Testing: normal, boundary and invalid cases
- **Final project:** Student Performance Management and Analysis System

| Previous | Next |
|---|---|
| [Module 15 — Python Data Handling](15_Python_Data_Handling.ipynb) | *(end of course — see the README for what to learn next)* |

---

**Learning objectives** — after this module you can:

1. Break a vague problem into clear, solvable pieces.
2. Write one solution two different ways and argue which is better.
3. Predict how a solution's work grows with input size — O(1), O(n), O(n²).
4. Pick the data structure a problem actually needs, and say why.
5. Write code another student can read, test it on purpose, and fix it
   methodically when it fails.

## Contents

1. [What is computational thinking?](#1)
2. [Abstraction](#2)
3. [Decomposition](#3)
4. [Generalisation](#4)
5. [Algorithmic thinking — two solutions compared](#5)
6. [Efficiency and Big-O in three steps](#6)
7. [Choosing data structures](#7)
8. [Maintainable code](#8)
9. [Testing](#9)
10. [Debugging as a discipline](#10)
11. [Final project — Student Performance Management and Analysis System](#11)
12. [Lab 16](#12)
13. [Common mistakes](#13)
14. [Exercises](#14)
15. [Summary and where to go next](#15)

<a id="1"></a>

## 1. What is computational thinking?

### Concept

**Computational thinking** is solving problems the way a computer
scientist solves them — *before* writing any code. It has four working
habits (Wing, 2006):

| Habit | Question it answers |
|---|---|
| **Decomposition** | Can I split this big problem into small ones? |
| **Abstraction** | Which details can I safely ignore right now? |
| **Pattern recognition / generalisation** | Have I solved something like this before? Does my solution work for *all* cases of this kind? |
| **Algorithm design** | What is the precise step-by-step recipe? |

Notice what is *absent*: a programming language. You have been doing
computational thinking since Module 3's temperature converter — every
"understand → inputs → outputs → pseudocode → implement → test" cycle
(Module 13) was an exercise in it. This module makes the habits explicit,
names them, and pushes them one level further.

### Example — the same problem, thought about three ways

Problem: *"Print how many students passed."*

- **Without thinking:** start typing immediately, get it wrong twice.
- **With decomposition:** load records (Module 15) → filter marks ≥ 50 →
  count them → print.
- **With generalisation:** "passed" is a *threshold question* — write
  `count_at_least(records, threshold)` once, and "how many got an A?"
  becomes the same function with 80.

In [ ]:
students = [
    {"name": "Ali", "marks": 78},
    {"name": "Sara", "marks": 91},
    {"name": "Ahmed", "marks": 65},
    {"name": "Omar", "marks": 45},
]


def count_at_least(records, threshold):
    """Count the records whose 'marks' are at least threshold."""
    count = 0
    for record in records:
        if record["marks"] >= threshold:
            count += 1
    return count


print("Passed (>= 50):", count_at_least(students, 50))
print("Got an A (>= 80):", count_at_least(students, 80))

One function, two questions. That is computational thinking paying rent.

<a id="2"></a>

## 2. Abstraction

### Concept

**Abstraction** means hiding details that do not matter at this moment,
so you can think about what does. Every tool you already use is an
abstraction:

- `len(students)` hides *how* Python counts.
- `sum(marks)` hides the loop you would otherwise write.
- `student["marks"]` hides where the data physically lives.
- `open(path)` hides buffers, encodings and the operating system.

When you write a function with a clear name and a docstring, **you are
building an abstraction for the next person** — usually future you.

### Why do we need it?

Your working memory holds roughly four things. A program that must be
understood all-at-once cannot fit in any head. Abstraction lets you
compress "these 12 lines do X" into one word, and think in words.

### Beginner example — naming a computation

In [ ]:
def is_pass(mark):
    """Return True when mark is enough to pass (50 or more)."""
    return mark >= 50


# The call sites read like sentences — the rule lives in ONE place.
print("Ali passed?  ", is_pass(78))
print("Omar passed? ", is_pass(45))

marks = [78, 91, 65, 45]
passed_count = 0
for mark in marks:
    if is_pass(mark):
        passed_count += 1
print("Passed count:", passed_count)

If the university ever changes the pass mark to 40, you edit **one
line** — not every `>= 50` scattered through the program. That is the
practical value of abstraction: *change in one place*.

### Practical example — layers of abstraction

A report program is layers calling layers; each layer only knows the
layer directly below:

In [ ]:
def average_of(marks):
    """Return the mean of a list of numbers."""
    return sum(marks) / len(marks)


def class_average(records):
    """Return the average mark across all records."""
    marks = []
    for record in records:
        marks.append(record["marks"])
    return average_of(marks)          # builds on the layer below


def print_summary(records):
    """Print one line summarising the records."""
    print("Students:", len(records), end=" ")
    print("Average:", round(class_average(records), 1))   # and the layer below that


records = [
    {"name": "Ali", "marks": 78},
    {"name": "Sara", "marks": 91},
    {"name": "Ahmed", "marks": 65},
]
print_summary(records)

`print_summary` neither knows nor cares *how* the average is computed —
that detail is one layer down. Reading it, you can trust each word.

### Challenge example — abstracting a *policy*, not just a computation

Abstraction also captures rules. Here the grading policy is data-driven
— one table controls the whole program:

In [ ]:
GRADE_TABLE = [
    (80, "A"),
    (60, "B"),
    (50, "C"),
    (0, "F"),
]


def grade_for(mark):
    """Return the letter grade for mark, using GRADE_TABLE."""
    for threshold, letter in GRADE_TABLE:
        if mark >= threshold:
            return letter
    return "F"


for mark in [95, 84, 72, 51, 39]:
    print(mark, "->", grade_for(mark))

Change the policy (new grade "A+" at 90?) and you edit the table — no
new `elif` chains, no other code touched. Data-driven abstraction is
the friendliest kind.

<a id="3"></a>

## 3. Decomposition

### Concept

**Decomposition** splits a big problem into smaller problems you can
solve separately, then composes the solutions. A 200-line `main()`
cannot be understood; ten 20-line functions can.

### Why do we need it?

Three separate wins:

1. **Understanding** — each piece fits in your head.
2. **Testing** — each function can be checked alone (§9).
3. **Teamwork** — different people can own different functions.

### Example — decomposing "process the mark sheet"

```text
Process the mark sheet
├── load_records(path)          read the CSV  (Module 11/15)
├── clean_records(rows)         fix mess      (Module 15)
├── compute_statistics(records) averages etc. (Module 15)
└── write_report(records, ...)  print + save  (Module 11)
```

Each box is a function you have *already written* in an earlier module.
Decomposition means recognising that the new big problem is composed of
old small ones.

### Beginner example — from one long script to four functions

In [ ]:
# The undecomposed way (works, but one tangled piece):
raw_lines = ["Ali 78", "Sara 91", "Ahmed 45"]

# ...everything at once: parse, convert, analyse, print...
parsed = []
for line in raw_lines:
    parts = line.split()
    parsed.append({"name": parts[0], "marks": int(parts[1])})
total = 0
for record in parsed:
    total += record["marks"]
print("Undecomposed average:", total / len(parsed))

It runs — but the parsing rule, the conversion and the statistics live
in one blob. The decomposed version:

In [ ]:
def parse_line(line):
    """Convert one 'Name Marks' line into a record."""
    parts = line.split()
    return {"name": parts[0], "marks": int(parts[1])}


def parse_lines(lines):
    """Convert every line into a record."""
    records = []
    for line in lines:
        records.append(parse_line(line))
    return records


def mean_mark(records):
    """Return the average marks of the records."""
    total = 0
    for record in records:
        total += record["marks"]
    return total / len(records)


records = parse_lines(raw_lines)
print("Decomposed average:", round(mean_mark(records), 2))
print("Records:           ", records)

**What improved:** `parse_line` can be tested with one weird line;
`mean_mark` never sees a string; adding a second file format means
writing one new *parse* function — nothing else changes.

### Practical example — decomposition with a function map

For the project below we will need a **top-level plan**. Writing the
function map first is how professionals start:

In [ ]:
# A function map as comments: the design BEFORE the code.
#
# Student Performance Management and Analysis System
# ├── load_students(path)          -> list of records        (Module 15)
# ├── add_student(records, rec)    -> updated list           (Module 7)
# ├── compute_stats(records)       -> summary dictionary     (Module 15)
# ├── grade_distribution(records)  -> {'A': n, ...}          (Module 8)
# ├── find_student(records, name)  -> record or None         (Module 15)
# ├── save_summary(summary, path)  -> JSON file              (Module 15)
# └── main()                       compose the above         (Module 9)
print("Function map printed as a comment — design first, code second.")

<a id="4"></a>

## 4. Generalisation

### Concept

**Generalisation** takes a solution that works for *one* case and reworks
it to work for *all* cases of that kind. The smell of un-generalised
code: a function called `count_passes()` and a near-twin
`count_A_grades()` and a near-twin `count_fails()`...

### Why do we need it?

Every near-copy multiplies the places a change must reach. One
general function beats four specific ones.

### Example — three twins become one function

In [ ]:
marks = [78, 91, 65, 45, 84]


# --- the un-generalised way: three near-identical loops ---
def count_passes(marks):
    """Count marks of 50 or more."""
    count = 0
    for mark in marks:
        if mark >= 50:
            count += 1
    return count


def count_A_grades(marks):
    """Count marks of 80 or more."""
    count = 0
    for mark in marks:
        if mark >= 80:
            count += 1
    return count


def count_fails(marks):
    """Count marks below 50."""
    count = 0
    for mark in marks:
        if mark < 50:
            count += 1
    return count


print(count_passes(marks), count_A_grades(marks), count_fails(marks))

Three loops, one condition different each time. The condition is the
*only* thing that varies — generalise exactly that:

In [ ]:
def count_where(marks, condition):
    """Count the marks for which condition(mark) is True."""
    count = 0
    for mark in marks:
        if condition(mark):
            count += 1
    return count


def at_least_50(mark):
    """Rule: pass means 50 or more."""
    return mark >= 50


def at_least_80(mark):
    """Rule: an A means 80 or more."""
    return mark >= 80


print("Passes:", count_where(marks, at_least_50))
print("A grades:", count_where(marks, at_least_80))

The generalisation passes *the varying part* as a value — here, a
function used as data (first-class functions, Module 14's mindset).
Without lambda syntax, plain helper functions do the job:

```python
def at_least_50(mark):
    return mark >= 50

def at_least_80(mark):
    return mark >= 80

count_where(marks, at_least_50)
count_where(marks, at_least_80)
```

Same generality, fully beginner-friendly. **Test yourself:** can you
rewrite `count_fails` as `count_where` with a helper? (That is Practice
item 2 in §14.)

<a id="5"></a>

## 5. Algorithmic thinking — two solutions compared

### Concept

An **algorithm** is a precise recipe. Almost every problem has *several*
correct recipes; algorithmic thinking means writing them down, comparing
their work, and choosing deliberately.

### Example 1 — "did anyone score 100?"

**Recipe A (scan):** look at every mark; remember if any equals 100.
**Recipe B (champion):** track the maximum; a maximum of 100 means
someone scored 100. Both are correct; A is simpler, B reuses a pattern
you know (Module 13's running maximum).

In [ ]:
marks = [78, 91, 65, 45, 84]

# Recipe A: scan with a flag.
has_hundred = False
for mark in marks:
    if mark == 100:
        has_hundred = True          # keep scanning — simplest to write
print("A: has 100?", has_hundred)

# Recipe B: running maximum.
highest = marks[0]
for mark in marks:
    if mark > highest:
        highest = mark
print("B: has 100?", highest == 100)

A third option — `100 in marks` — is one line and *also* correct,
because it is Recipe A inside a built-in (Modules 7–8). "Know the
manual" is part of algorithmic thinking: built-ins are battle-tested
scans.

### Example 2 — "find the two highest marks" (harder)

Try it yourself before reading: how would *you* do it?

**Recipe A:** sort a copy, take the last two. Short, and sorting costs
more work than needed for just two values — but with 5 marks who cares?
**Recipe B:** one pass tracking the top two. More code, less work,
scales to "top ten" naturally.

In [ ]:
marks = [78, 91, 65, 45, 84]

# Recipe A: sort a copy (sort() returns None, so copy first — Module 7).
sorted_copy = marks.copy()
sorted_copy.sort()
top_two_A = [sorted_copy[-1], sorted_copy[-2]]
print("A:", top_two_A)

# Recipe B: one pass, two champions.
best = marks[0]
second = None
for mark in marks[1:]:
    if mark > best:
        second = best
        best = mark
    elif second is None or mark > second:
        second = mark
top_two_B = [best, second]
print("B:", top_two_B)

Both give `[91, 84]`. For 5 marks, pick A and go to lunch. For 5
million marks, B's single pass matters. **Choosing is the skill** — and
the choice depends on n, which is exactly what the next section makes
precise.

### Challenge example — same problem, genuinely different algorithms

*"Is a word a palindrome?"* — two honest approaches:

In [ ]:
def is_palindrome_reverse(word):
    """Compare the word with its reversal (Module 4 slicing)."""
    return word == word[::-1]


def is_palindrome_two_ends(word):
    """Walk two ends towards the middle, comparing pairs."""
    left = 0
    right = len(word) - 1
    while left < right:
        if word[left] != word[right]:
            return False
        left += 1
        right -= 1
    return True


for word in ["level", "python"]:
    print(word, is_palindrome_reverse(word), is_palindrome_two_ends(word))

The reverse version does full-length copying work; the two-ends version
can stop early (it stops at the first mismatch — see `python`, which
fails on the very first pair). Same answer, different work profile.

<a id="6"></a>

## 6. Efficiency and Big-O in three steps

### Concept

**Efficiency** asks: how does the work grow when the input grows?
Computer scientists answer with **Big-O notation** — a *growth class*,
not a stopwatch:

| Big-O | Name | Example shape |
|---|---|---|
| **O(1)** | constant | same work however big the input: `numbers[0]`, `marks_dict["Ali"]` |
| **O(n)** | linear | one pass over n items: a single `for` loop |
| **O(n²)** | quadratic | for each of n items, loop over n items: nested loops |

Two warnings before the examples:

1. Big-O describes **growth**, not seconds. An O(n) loop can be slower
   in seconds than an O(n²) one *for small n* — the classes predict
   what happens when n becomes large.
2. We drop constants: `3n + 5` comparisons are still "O(n)". We care
   about the *shape of growth*.

### Step 1 — O(1): constant work

In [ ]:
numbers = list(range(1, 101))          # 100 numbers — but nothing about
#                                       the next two lines CARES how many.
first = numbers[0]                     # O(1): jump straight to index 0
last = numbers[-1]                     # O(1): jump straight to the end
print("first:", first, "last:", last)

Indexing never inspects the other 98 numbers. Doubling the list changes
nothing about the work in these lines — that is O(1).

### Step 2 — O(n): one pass

In [ ]:
total = 0
for number in numbers:                 # touches each of the n items once
    total += number                    # -> O(n)
print("Sum of 1..100:", total)

Double the list → roughly double the work → O(n). (Fun aside: Gauss
would compute this in O(1) with `n * (n + 1) / 2` — a better *algorithm*,
not better code. Algorithm choice beats micro-optimisation.)

### Step 3 — O(n²): a loop inside a loop

In [ ]:
pairs_seen = 0
for a in numbers[:5]:                  # outer: 5 times...
    for b in numbers[:5]:              # ...inner: 5 times each -> 25 steps
        pairs_seen += 1
print("Pairs for n=5: ", pairs_seen)

# If the outer loop ran over all 100 numbers, the inner would run 100
# times per outer step: 100 * 100 = 10,000 steps. That is O(n^2) growth.

**Reading growth off the table:** at n = 1,000, an O(n) pass does
~1,000 steps; O(n²) does ~1,000,000. At n = 1,000,000: 10⁶ versus 10¹².
Growth classes are why "it works on my 10-row file" is not the end of
the story.

### Practical example — finding duplicates two ways

The pair-scan below is the classic O(n²) pattern; the set version is
O(n) because set membership is O(1) (next section explains why):

In [ ]:
def has_duplicate_pairs(items):
    """Compare every pair — O(n^2): for each item, scan the rest."""
    for i in range(len(items)):
        for j in range(len(items)):
            if i != j and items[i] == items[j]:
                return True
    return False


def has_duplicate_set(items):
    """Remember seen items in a set — O(n): one pass, O(1) lookups."""
    seen = set()
    for item in items:
        if item in seen:
            return True
        seen.add(item)
    return False


small = [3, 1, 4, 1, 5]
print("pairs:", has_duplicate_pairs(small), " set:", has_duplicate_set(small))

Same answer, different growth. (Bonus: `len(set(items)) < len(items)`
answers it in one line — generalisation again.)

### Challenge example — measure the growth yourself

Count the steps both functions take as the input doubles:

In [ ]:
def steps_pairs(n):
    """Steps for the O(n^2) pair scan on n items."""
    return n * n


def steps_set(n):
    """Steps for the O(n) set scan on n items (1 pass, O(1) checks)."""
    return n


for n in [100, 200, 400, 800]:
    print(f"n={n:>4}  pairs: {steps_pairs(n):>7}   set: {steps_set(n):>4}")

Read the pattern: when n doubles, the O(n) count doubles — but the
O(n²) count **quadruples**. That ×2-vs-×4 signature is all Big-O is
saying. (These "steps" are simplified models — real timings vary — but
the *shape* is exactly right.)

<a id="7"></a>

## 7. Choosing data structures

### Concept

You now own four containers. The skill is matching container to job:

| Structure | Best at | Weak at | Typical job |
|---|---|---|---|
| **list** | order, position access (O(1)), appending | slow membership (O(n)) | sequences: marks, to-do items |
| **tuple** | fixed shape, safety (immutable) | any change | coordinates, RGB colours, (name, age) pairs |
| **set** | O(1) membership, uniqueness, algebra | no order, no positions | "have I seen X?", unique IDs |
| **dict** | O(1) lookup *by key*, mapping | no position numbers | records, counters, `roll_no -> name` |

### Why do we need it?

Structure choice is *algorithm* choice in disguise. The duplicate check
became O(n) precisely because `item in seen` on a **set** is O(1)
instead of the list's O(n).

### Example 1 — membership: list vs set

The correctness is identical; the growth is not:

In [ ]:
attendance_list = ["Ali", "Sara", "Ahmed", "Hina", "Omar", "Zoya"]
attendance_set = {"Ali", "Sara", "Ahmed", "Hina", "Omar", "Zoya"}

print("Was Ali there? (list)", "Ali" in attendance_list)   # correct, but O(n)
print("Was Ali there? (set) ", "Ali" in attendance_set)    # correct, and O(1)

For 6 names either works. For 6 million, the list scan hurts. Rule of
thumb: **many membership questions → set**.

### Example 2 — lookup by key: list-scan vs dictionary

Finding a student's marks by roll number:

In [ ]:
students = [
    {"roll_no": "DS-101", "name": "Ali", "marks": 78},
    {"roll_no": "DS-102", "name": "Sara", "marks": 91},
    {"roll_no": "DS-103", "name": "Ahmed", "marks": 65},
]

# Approach 1: scan the list each time — O(n) per lookup.
def marks_by_scan(records, roll_no):
    """Find marks by scanning — fine once, painful in a loop."""
    for record in records:
        if record["roll_no"] == roll_no:
            return record["marks"]
    return None


# Approach 2: build ONE dictionary, then look up forever — O(1) per lookup.
marks_by_roll = {}
for record in students:
    marks_by_roll[record["roll_no"]] = record["marks"]

print("scan:", marks_by_scan(students, "DS-102"))
print("dict:", marks_by_roll["DS-102"])

Rule of thumb: **repeated "find the one thing with this ID" →
dictionary**, built once. (In a big system you would have both: the
list for ordered processing, the dict as its index.)

### Example 3 — fixed-shape data: tuple

A point in the plane never grows a third coordinate mid-program. A
tuple says so *and* enforces it:

In [ ]:
point = (3, 4)                      # x, y — fixed by design
red = (255, 0, 0)                   # RGB — same idea

x, y = point                        # unpacking (Module 8)
print("Point:", x, y)

# point[0] = 5                       # uncommenting this raises:
#                                    # TypeError: 'tuple' object does not
#                                    # support item assignment
print("Distance-squared from origin:", x * x + y * y)

Rule of thumb: **the shape will never change → tuple**; the code
documents itself and accidents become loud errors.

### Practical example — a mini "structure chooser" decision table

Run this and keep it as a revision card:

In [ ]:
questions = [
    ("Does order + position matter?",            "list"),
    ("Is it a fixed shape that never changes?",  "tuple"),
    ("Only membership/uniqueness questions?",    "set"),
    ("Look up values by a permanent key?",       "dict"),
]
for question, answer in questions:
    print(f"{question:<42} -> {answer}")

<a id="8"></a>

## 8. Maintainable code

### Concept

**Maintainable code** is code the *next* reader can change safely. The
next reader is usually you, three weeks later, with no memory. The
habits are small and boring, which is why they work:

1. **Meaningful names** — `passed_count`, not `p`; `is_valid`, not `flag2`.
2. **Small functions** — one job each; a docstring saying what it returns.
3. **No duplication** — the rule lives in one place (abstraction!).
4. **Comments where useful** — the *why*, not the *what*.
5. **Consistent formatting** — PEP 8: 4-space indents, spaces around
   operators, `snake_case` for variables and functions.
6. **Logical organisation** — constants at the top, functions next,
   the "main" flow at the bottom (the Module 10 script layout).

### Example — a small refactoring story

Same behaviour, three versions of quality:

In [ ]:
# Version 1 — works, but the next reader suffers.
def f(l):
    t = 0
    for x in l:
        t = t + x
    return t / len(l)


# Version 2 — better names, clearer flow.
def average(numbers):
    total = 0
    for number in numbers:
        total += number
    return total / len(numbers)


# Version 3 — add a docstring and guard the empty case (Module 9 style).
def average_safe(numbers):
    """Return the mean of numbers, or 0.0 for an empty list."""
    if len(numbers) == 0:
        return 0.0
    total = 0
    for number in numbers:
        total += number
    return total / len(numbers)


marks = [78, 91, 65]
print(round(f(marks), 1), round(average(marks), 1), round(average_safe(marks), 1))
print("Empty list:", average_safe([]))

The jump from 1→2 costs nothing; from 2→3 costs one `if`. Both jumps
save the next reader real time — and version 3 cannot crash on the
empty-list case that version 1 would turn into `ZeroDivisionError`.

### Practical example — comments that earn their place

In [ ]:
def apply_discount(price, percent):
    """Return price after reducing it by percent (0-100).

    Invalid percents are clamped to the valid range rather than
    rejected, because callers here pass user-typed values.
    """
    if percent < 0:
        percent = 0            # a negative discount would INCREASE the price
    if percent > 100:
        percent = 100          # above 100 would produce a negative price
    return price * (100 - percent) / 100


print(apply_discount(500, 20))
print(apply_discount(500, 150))     # clamped, not crashed
print(apply_discount(500, -5))      # clamped, not crashed

The comments explain **why** the clamps exist (the *what* is visible).
That is the standard: delete any comment that only restates the code.

<a id="9"></a>

## 9. Testing

### Concept

**Testing** means running code against inputs whose right answer you
know *before* looking at the output. A **test case** has three parts:

```text
input -> expected output     (written down BEFORE running)
```

Three kinds of input every function deserves:

| Kind | Example for `average` |
|---|---|
| **normal** | `[78, 91, 65]` → `78.0` |
| **boundary** | `[50]` (single), `[]` (empty) |
| **invalid** | — whatever your design declares out of contract |

### Why do we need it?

"It printed something sensible once" is not evidence. Tests are
evidence you can re-run — after every change, forever.

### Example — manual test cases

A test function per rule: compare expected with actual and *say so*:

In [ ]:
def average_safe(numbers):
    """Return the mean of numbers, or 0.0 for an empty list."""
    if len(numbers) == 0:
        return 0.0
    total = 0
    for number in numbers:
        total += number
    return total / len(numbers)


def check_average(numbers, expected):
    """One test case: compare expected with actual, print the verdict."""
    actual = average_safe(numbers)
    if actual == expected:
        print("PASS:", numbers, "->", actual)
    else:
        print("FAIL:", numbers, "-> expected", expected, "got", actual)


check_average([78, 91, 65], 78.0)      # normal
check_average([10, 20, 30, 40], 25.0)  # normal, whole average
check_average([50], 50.0)              # boundary: single value
check_average([], 0.0)                 # boundary: empty

**Explanation:** each `check_average` call is one test case with its
expected value *written into the code*. When someone later "improves"
`average_safe` and breaks it, the same four lines catch the regression
immediately. (Python's `assert` statement can replace the print; the
explicit version shows you exactly what automated test frameworks
automate.)

### Practical example — testing a classifier on boundaries

Grade boundaries are where classifiers die. Test *at* each boundary:

In [ ]:
def grade_for(mark):
    """A >= 80, B >= 60, C >= 50, else F."""
    if mark >= 80:
        return "A"
    if mark >= 60:
        return "B"
    if mark >= 50:
        return "C"
    return "F"


def check_grade(mark, expected):
    actual = grade_for(mark)
    verdict = "PASS" if actual == expected else "FAIL"
    print(f"{verdict}: grade_for({mark}) = {actual} (expected {expected})")


check_grade(95, "A")     # comfortably inside
check_grade(80, "A")     # boundary: exactly at the A line
check_grade(79, "B")     # boundary: one below the A line
check_grade(60, "B")     # boundary: exactly at the B line
check_grade(50, "C")     # boundary: exactly at the C line
check_grade(49, "F")     # boundary: one below the C line
check_grade(0, "F")      # extreme

Notice the deliberate pattern: for each threshold, test *exactly at it*
and *one below it*. Off-by-one bugs cannot survive that pair.

### Challenge example — a tiny test suite for a real function

Bundle the cases so one run tests everything:

In [ ]:
def is_strong_password(text):
    """A strong password has 8+ characters and at least one digit."""
    if len(text) < 8:
        return False
    has_digit = False
    for character in text:
        if character.isdigit():
            has_digit = True
    return has_digit


test_cases = [
    ("abcdefg1", True),      # 8 chars incl. digit — normal
    ("abcdefgh", False),     # 8 chars, no digit — boundary kind
    ("abc1", False),         # too short — boundary: length
    ("12345678", True),      # all digits still counts — surprising but valid
    ("", False),             # empty — boundary
]

for text, expected in test_cases:
    actual = is_strong_password(text)
    verdict = "PASS" if actual == expected else "FAIL"
    print(f"{verdict}  is_strong_password({text!r}) -> {actual}")

A list of `(input, expected)` pairs + one loop = the skeleton of every
professional test suite. When you meet the `pytest` framework in later
courses, you will recognise the shape instantly.

<a id="10"></a>

## 10. Debugging as a discipline

### Concept

You have debugged since Module 1. What changes now is *method*. The
professional loop:

```text
1. Reproduce   make the bug happen on demand, with the smallest input possible
2. Localise    binary-search your program: which HALF contains the bug?
3. Inspect     print() the variables around the suspect line
4. Fix         change ONE thing
5. Re-test     run the tests from §9 — including the case that failed
```

### Example — localising with prints, then removing them

A buggy function: intended to return the average of the *even* numbers.

In [ ]:
def average_of_evens(numbers):
    """Return the average of the even numbers in the list.

    Known buggy: returns the wrong value for [1, 2, 4].
    """
    total = 0
    count = 0
    for number in numbers:
        if number % 2 == 0:
            total += number
        count += 1          # BUG: counts EVERY number, not only evens
    return total / count


print(average_of_evens([1, 2, 4]))     # correct answer is 3.0; got 2.0

Now *localise*: is `total` wrong, or `count`? Print both inside the loop:

In [ ]:
def average_of_evens_debug(numbers):
    """Same function with temporary debug prints."""
    total = 0
    count = 0
    for number in numbers:
        if number % 2 == 0:
            total += number
        count += 1
        print("  number:", number, "total:", total, "count:", count)  # DEBUG
    return total / count


print(average_of_evens_debug([1, 2, 4]))
print("The trace shows count reaching 3 — it must only count evens.")

The trace pinpoints it: `count` increments even for `1`. Fix — move it
inside the `if` — then **keep a regression test**:

In [ ]:
def average_of_evens_fixed(numbers):
    """Return the average of the even numbers, or None if there are none."""
    total = 0
    count = 0
    for number in numbers:
        if number % 2 == 0:
            total += number
            count += 1            # FIX: only evens are counted
    if count == 0:
        return None               # FIX 2: no evens -> no division by zero
    return total / count


cases = [
    ([1, 2, 4], 3.0),             # the original failing case
    ([1, 3, 5], None),            # no evens at all
    ([2], 2.0),                   # single even
]
for numbers, expected in cases:
    actual = average_of_evens_fixed(numbers)
    verdict = "PASS" if actual == expected else "FAIL"
    print(verdict, numbers, "->", actual)

The debugging cycle closed: reproduce → trace → fix one thing →
re-test (and the fix exposed a *second* bug — the empty case — which
is exactly why re-testing matters).

<a id="11"></a>

## 11. Final project — Student Performance Management and Analysis System

Everything from Modules 7–16 in one program. We build it in the order
of Module 13's methodology: problem → requirements → data design →
algorithm → pseudocode → implementation → testing → improvements.

### 1. Problem statement

A course office keeps student marks in a CSV file. They need a small
system that loads the data, keeps it consistent, answers common
questions (per-student, per-grade, whole-class), and saves a summary
that other tools can read.

### 2. Requirements

- Load students from `students.csv` (`name,roll_no,marks`), rejecting
  invalid rows with a *reason*.
- Add new students interactively (validated — Module 12).
- Answer queries: find by roll number, grade distribution, class
  statistics, top and at-risk students.
- Save the summary to `summary.json`.
- Everything in core Python + standard library.

### 3. Input / output specification

```text
INPUT   students.csv   header: name,roll_no,marks
MENU    user choices 1-6 (validated; '6' quits)
OUTPUT  console report + summary.json
```

### 4. Data design

```text
students      : list of records  {'name': str, 'roll_no': str, 'marks': int}
roll_index    : dict  roll_no -> record      (O(1) lookup — §7!)
used_ids      : set   of roll_nos already taken (O(1) uniqueness — §7!)
summary       : dict  {'count', 'average', 'highest', 'lowest'}
grades        : dict  {'A': n, 'B': n, 'C': n, 'F': n}
```

Notice how §7's decision table made the choices: dictionary for
lookups, set for uniqueness, list for ordered records.

### 5. Algorithm

```text
1. load   read CSV -> clean rows -> records, rejections
2. index  build roll_index (dict) and used_ids (set)
3. loop   show menu -> read choice (validated) -> run action
          1 report   2 add student   3 find by roll
          4 statistics   5 grade chart   6 save & quit
4. save   write summary.json before quitting
```

### 6. Pseudocode

```text
MAIN:
  records, rejected = LOAD("students.csv")
  roll_index = {r.roll_no: r for records}      # loop, not comprehension!
  used_ids = set of roll_nos
  REPEAT
    choice = ASK_MENU()
    IF choice = 1: PRINT_REPORT(records)
    IF choice = 2: ADD_STUDENT(records, roll_index, used_ids)
    IF choice = 3: FIND_AND_SHOW(roll_index)
    IF choice = 4: SHOW_STATISTICS(records)
    IF choice = 5: SHOW_GRADE_CHART(records)
  UNTIL choice = 6
  SAVE_SUMMARY(records, "summary.json")
```

### 7. Program structure

```text
project/
├── students.csv          input data
├── analyzer.py           constants, then functions, then main()
└── summary.json          produced by choice 6
```

### 8. Implementation — stage by stage

**Stage A — sample data** (as the office's CSV export would look):

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("file_sandbox") / "final_project"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DATA_FILE = PROJECT_DIR / "students.csv"
SUMMARY_FILE = PROJECT_DIR / "summary.json"

DATA_FILE.write_text(
    "name,roll_no,marks\n"
    "Ali,DS-101,78\n"
    "Sara,DS-102,91\n"
    "Ahmed,DS-103,65\n"
    "Hina,DS-104,84\n"
    "Omar,DS-105,45\n"
    "Zoya,DS-106,72\n"
    "Bilal,DS-107,abc\n"       # will be rejected: not numeric
    ",DS-108,55\n",            # will be rejected: no name
    encoding="utf-8",
)
print("Data file ready:", DATA_FILE.as_posix())

**Stage B — loading with rejection reasons** (Module 15's cleaner,
upgraded to explain *why* each row was refused):

In [ ]:
import csv


def clean_row(row):
    """Return (record, None) for a valid row, or (None, reason)."""
    name = row["name"].strip().title()
    marks_text = row["marks"].strip()
    if name == "":
        return None, "missing name"
    if marks_text == "":
        return None, "missing marks"
    if not marks_text.isdigit():
        return None, "marks not numeric"
    marks = int(marks_text)
    if marks < 0 or marks > 100:
        return None, "marks out of range"
    return {"name": name, "roll_no": row["roll_no"].strip(), "marks": marks}, None


def load_students(path):
    """Load records from path; return (records, rejected) where rejected
    is a list of (row, reason) pairs."""
    records = []
    rejected = []
    with open(path, "r", newline="", encoding="utf-8") as file:
        for row in csv.DictReader(file):
            record, reason = clean_row(row)
            if record is None:
                rejected.append((row, reason))
            else:
                records.append(record)
    return records, rejected


students, rejected_rows = load_students(DATA_FILE)
print("Loaded", len(students), "students")
for row, reason in rejected_rows:
    print("Rejected", row, "->", reason)

**Stage C — the index and the uniqueness set** (the §7 choices, in
action):

In [ ]:
roll_index = {}                 # roll_no -> record: O(1) find
used_ids = set()                # every roll_no taken: O(1) uniqueness check
for record in students:
    roll_index[record["roll_no"]] = record
    used_ids.add(record["roll_no"])

print("Index size:", len(roll_index), " Set size:", len(used_ids))
print("DS-102 is:", roll_index["DS-102"]["name"])

**Stage D — analysis functions** (Module 15's accumulators, packaged):

In [ ]:
def compute_stats(records):
    """Return {'count', 'average', 'highest', 'lowest'} for records."""
    if len(records) == 0:
        return {"count": 0, "average": None, "highest": None, "lowest": None}
    marks = []
    for record in records:
        marks.append(record["marks"])
    return {
        "count": len(marks),
        "average": round(sum(marks) / len(marks), 1),
        "highest": max(marks),
        "lowest": min(marks),
    }


def grade_distribution(records):
    """Return {'A': n, 'B': n, 'C': n, 'F': n} (A>=80, B>=60, C>=50)."""
    counts = {"A": 0, "B": 0, "C": 0, "F": 0}
    for record in records:
        marks = record["marks"]
        if marks >= 80:
            counts["A"] += 1
        elif marks >= 60:
            counts["B"] += 1
        elif marks >= 50:
            counts["C"] += 1
        else:
            counts["F"] += 1
    return counts


def top_and_at_risk(records):
    """Return (top names >=80, at-risk names <50)."""
    top = []
    at_risk = []
    for record in records:
        if record["marks"] >= 80:
            top.append(record["name"])
        elif record["marks"] < 50:
            at_risk.append(record["name"])
    return top, at_risk


summary = compute_stats(students)
grades = grade_distribution(students)
top, at_risk = top_and_at_risk(students)
print("Stats: ", summary)
print("Grades:", grades)
print("Top:   ", top, " At risk:", at_risk)

**Stage E — the interactive actions** (adding a student uses Module 12
validation; the menu loop will be scripted in the tests below):

In [ ]:
def add_student(records, roll_index, used_ids, name, roll_no, marks):
    """Validate and add one student; return an error message or None."""
    if name.strip() == "":
        return "name is empty"
    if roll_no.strip() == "":
        return "roll number is empty"
    if roll_no in used_ids:
        return "roll number already used"
    if marks < 0 or marks > 100:
        return "marks must be 0-100"
    record = {"name": name.strip().title(), "roll_no": roll_no.strip(), "marks": marks}
    records.append(record)
    roll_index[record["roll_no"]] = record     # keep the index in sync
    used_ids.add(record["roll_no"])            # and the uniqueness set
    return None


error = add_student(students, roll_index, used_ids, "nida", "DS-109", 88)
print("Add 'nida':", error if error else "added OK")
error = add_student(students, roll_index, used_ids, "Fake", "DS-102", 50)
print("Add duplicate roll:", error)

**Stage F — the report and the save:**

In [ ]:
def print_report(records, rejected_rows):
    """Print the full student report."""
    print("=" * 44)
    print("STUDENT PERFORMANCE MANAGEMENT SYSTEM")
    print("=" * 44)
    print(f"{'Name':<8}{'Roll':<10}{'Marks':>5}")
    print("-" * 44)
    for record in records:
        print(f"{record['name']:<8}{record['roll_no']:<10}{record['marks']:>5}")
    print("-" * 44)
    print("Loaded:", len(records), " Rejected:", len(rejected_rows))


def save_summary(records, path):
    """Compute and save the summary as JSON; return the summary."""
    stats = compute_stats(records)
    grades = grade_distribution(records)
    top, at_risk = top_and_at_risk(records)
    full_summary = {"statistics": stats, "grades": grades,
                    "top": top, "at_risk": at_risk}
    with open(path, "w", encoding="utf-8") as file:
        import json
        json.dump(full_summary, file, indent=4)
    return full_summary


print_report(students, rejected_rows)
saved = save_summary(students, SUMMARY_FILE)
print("Saved keys:", list(saved.keys()))

**Stage G — the menu loop.** Below it is driven by a *scripted* session
(choice 4, then 6) so the notebook runs unattended; on your machine the
same `main()` asks `input()` in a terminal exactly as in Module 3.

In [ ]:
def main():
    """Run one scripted menu session (swap input() in for interactive use)."""
    session = ["4", "6"]           # scripted choices: show stats, then quit
    step = 0
    students, rejected_rows = load_students(DATA_FILE)
    roll_index = {}
    used_ids = set()
    for record in students:
        roll_index[record["roll_no"]] = record
        used_ids.add(record["roll_no"])

    while True:
        # choice = input("Menu 1 report 2 add 3 find 4 stats 5 grades 6 quit: ")
        choice = session[step]     # scripted instead of input() for the notebook
        step += 1
        if choice == "4":
            print("Stats:", compute_stats(students))
        elif choice == "6":
            save_summary(students, SUMMARY_FILE)
            print("Bye — summary saved to", SUMMARY_FILE.as_posix())
            break


main()

**9. Testing the project.** The test-first table from §9, applied to
the system's core functions:

In [ ]:
cases = [
    # (function input: records list,   expected stats dict)
    ([{"name": "A", "roll_no": "R1", "marks": 50}], "single"),
    ([], "empty"),
    ([{"name": "A", "roll_no": "R1", "marks": 70},
      {"name": "B", "roll_no": "R2", "marks": 90}], "two records"),
]
for records, label in cases:
    stats = compute_stats(records)
    print(f"{label:>12}: count={stats['count']} average={stats['average']}")

And the boundary tests for the add-student validator:

In [ ]:
def check_add(name, roll_no, marks, expected_error):
    """One add-student test case against a fresh, empty system."""
    test_records = []
    test_index = {}
    test_used = set()
    actual_error = add_student(test_records, test_index, test_used, name, roll_no, marks)
    verdict = "PASS" if actual_error == expected_error else "FAIL"
    print(f"{verdict}  add({name!r}, {roll_no!r}, {marks}) -> {actual_error!r}")


check_add("Ali", "DS-001", 50, None)          # normal
check_add("", "DS-002", 50, "name is empty")  # invalid name
check_add("Sara", "", 50, "roll number is empty")
check_add("Sara", "DS-003", 101, "marks must be 0-100")   # boundary: above
check_add("Sara", "DS-004", -1, "marks must be 0-100")    # boundary: below
check_add("Sara", "DS-005", 0, None)          # boundary: exactly 0 is valid
check_add("Sara", "DS-006", 100, None)        # boundary: exactly 100 is valid

**10. Error handling in the project** — what each layer guarantees:

| Failure | Handled by |
|---|---|
| `students.csv` missing | `try/except FileNotFoundError` around load (Module 12) |
| dirty row | `clean_row` returns a *reason*, never crashes the load |
| duplicate roll number | `used_ids` set check before inserting |
| out-of-range marks | range check in `add_student` and `clean_row` |
| empty dataset | `compute_stats` returns `None` fields, report prints gracefully |

**11. Sample data & 12. sample output** — you ran both above. **13.
Possible improvements** (each maps to a *future* module, not a mystery):

- Per-subject marks: records become
  `{"name": ..., "marks": {"math": ..., "cs": ...}}` (nested dicts).
- Sort the report by marks (Module 7 `sort(key=...)`).
- Let a `Student` class own the per-record logic (Module 14).
- Move loaders/analysers into a `school` package (Module 10).
- Rejection log to `rejected.csv` (Module 11).
- Replace the scripted menu with real `input()` + validation loops
  (Modules 3 and 12).

<a id="12"></a>

## 12. Lab 16 — Mini Project

**Learning objectives:** apply the full computational-thinking cycle to
a problem of your own; design before coding; test on purpose.

**Theory recap (5 minutes):** decompose → abstract → generalise →
choose structures → implement small functions → test boundaries →
debug by method.

### Warm-up (10 minutes)

1. Write `count_vowels(text)` with a loop; name the pattern it uses.
2. State the Big-O of your function and of `text.count("a")`.
3. Which structure would you use for "which users liked which posts"?
   Why?

### Guided exercise

**G1.** With your instructor: take "print a class attendance sheet from
a CSV" through all ten methodology steps of Module 13 on the
whiteboard, then implement it.

### Independent exercises

**I1.** Refactor *your* Module 13 menu program: extract every block
into a named function; keep behaviour identical. Verify with the §9
test pattern.

**I2.** Take one O(n²) loop you have written this semester and rewrite
it with a set or dictionary. Measure the step counts for n = 100,
200, 400 like §6's challenge.

**I3.** Write the `(input, expected)` test table for
`is_strong_password` with *five* new cases, including one you expect
to FAIL before you run it — then investigate what actually happens.

### Challenge problems

**C1.** **Library loans** — books.csv (`title,author,copies`) and
loans.csv (`student_id,title`): build a system that can lend, return,
and report who holds what, using dict + set correctly.

**C2.** **Text analyser** — load a text file; report the top-10 most
frequent words (a dictionary counter + one sort), the longest word,
and the Big-O of each part of your solution.

**C3.** **Dashboard** — combine the final project's JSON summary with
a text "chart": print each grade as `A: ****` bars, sorted most-common
first.

**Expected learning outcomes:** you can now carry a problem from a
vague sentence to tested, organised, structure-appropriate code — and
justify every choice on the way.

<a id="13"></a>

## 13. Common mistakes

```python
# 1. Confusing growth classes with speed for small inputs.
# O(n^2) with n=5 finishes instantly. Big-O predicts growth, not seconds.
```

```python
# 2. Membership tests in a loop over a LIST.
for item in new_items:
    if item in huge_list:          # O(n) each time -> O(n*m) total
        ...
# Fix: huge_set = set(huge_list)   then  if item in huge_set:  (O(1) each)
```

```python
# 3. Sorting to get one maximum.
biggest = sorted(marks)[-1]        # correct, but sorts EVERYTHING (O(n log n))
# Fix: max(marks)                 — one pass, clearer intent.
```

```python
# 4. Names that hide meaning.
d2 = {}
for x in l1:
    d2[x[0]] = x[1]
# Fix: name the roles, not the containers:
#      marks_by_roll[record["roll_no"]] = record["marks"]
```

```python
# 5. Testing only the happy path.
check_average([70, 80], 75.0)      # fine...
# ...and never testing [] -> the ZeroDivisionError ships to production.
# Fix: always add boundary cases (empty, single, exact-threshold).
```

```python
# 6. Copy-paste "reuse".
# Two 30-line functions differing in one number will drift apart.
# Fix: extract the shared part; pass the difference as a parameter.
```

<a id="14"></a>

## 14. Exercises

### Quick Review

1. Name the four habits of computational thinking and one question each
   answers.
2. Your friend says "O(1) is always faster than O(n)". What is right
   and what is missing from that claim?
3. What does `numbers[0]` have to do with O(1)?
4. Why does putting the seen-items in a *set* make the duplicate check
   O(n) instead of O(n²)?
5. Give one job where a tuple beats a list, and say why.
6. What three kinds of test input does every function deserve?
7. In the debugging loop, why fix only ONE thing at a time?
8. What is wrong with a comment saying `# add 1 to count`?
9. Why is `max(marks)` preferable to `sorted(marks)[-1]`?
10. In the final project, why keep BOTH the records list and the
    roll-number dictionary?

### Predict the Output

```python
# 1.
def mystery(n):
    total = 0
    for i in range(n):
        for j in range(n):
            total += 1
    return total
print(mystery(4))
```

```python
# 2.
values = [4, 8, 15, 16, 23, 42]
biggest = values[0]
for value in values:
    if value > biggest:
        biggest = value
print(biggest)
```

```python
# 3.
seen = set()
for item in ["a", "b", "a", "c"]:
    seen.add(item)
print(len(seen))
```

```python
# 4.
data = {"x": 1, "y": 2}
data["z"] = data["x"] + data["y"]
print(sorted(data.keys()), data["z"])
```

```python
# 5.
def f(word):
    return word == word[::-1]
print(f("racecar"), f("racer"))
```

<details>
<summary>Predict the Output — answers (click to expand)</summary>

1. `16` — 4×4 inner steps (O(n²), and a hint at why tables of n² grow
   so fast).
2. `42` — the running-champion pattern.
3. `3` — sets discard duplicates.
4. `['x', 'y', 'z'] 3`
5. `True False`
</details>

### Find the Error

```python
# 1. Intent: average without crashing on empty input.
def average(numbers):
    if numbers == []:
        return 0
    return sum(numbers) / len(numbers)
print(average())
```

```python
# 2. Intent: count how many marks are 50 or more.
marks = [45, 50, 78, 49, 50]
count = 0
for mark in marks:
    if mark > 50:
        count += 1
print(count)
```

```python
# 3. Intent: unique names, in a set.
names = {"Ali", "Sara", "Ali"}
print(names[0])            # why does this line fail?
```

```python
# 4. Intent: find whether any mark is exactly 100.
marks = [80, 100, 95]
found = False
for mark in marks:
    if mark == 100:
        found = True
        break
        print("Found!")     # the student says this never prints — who is wrong?
print(found)
```

```python
# 5. Intent: one test case for a boundary.
def is_adult(age):
    return age >= 18

print(is_adult(18), is_adult(17))   # outputs 'True False' — so it works?
# The test PASSED, but the NEXT function call crashes. Which call would,
# and what kind of input was forgotten?
```

<details>
<summary>Find the Error — answers (click to expand)</summary>

1. `average()` is called with **no argument**, but the function requires
   `numbers` — `TypeError`. Fix: `average([])`.
2. `>` should be `>=`: the marks of exactly 50 are excluded, so it
   prints 1 instead of 3. Classic off-by-one at a threshold.
3. Sets have **no positions** — indexing is undefined
   (`TypeError: 'set' object is not subscriptable`). Convert to a sorted
   list if you need positions: `sorted(names)[0]`.
4. The `print` after `break` is unreachable — dead code. The student is
   right that it never prints; move the print before the `break` (or
   delete it). `found` still prints `True`.
5. The forgotten kind is the **non-numeric/invalid** input:
   `is_adult("eighteen")` crashes with `TypeError` (`str >= int`). The
   two numeric boundary cases pass, which proves only that numbers work.
</details>

### Programming Exercises

1. **Growth table** — print n, n², n·log₂(n) for n = 10, 100, 1000
   (use `math.log2`) and comment on which column frightens you.
2. **Generalise** — write `count_where(items, condition)` using a helper
   function per rule; implement three different rules with it.
3. **Two-sum** — for a list and a target, return the *first* pair
   summing to the target: write the O(n²) pair-scan, then the O(n)
   version using a set of seen complements.
4. **Duplicate reporter** — return the *list of duplicated values* from
   a list, once each, using a set; state your solution's Big-O.
5. **Boundaries** — write `is_valid_roll(roll_no)` (format `DS-###`,
   digits 001-999) and a complete boundary test table for it.
6. **Refactor** — take any earlier module's script-style program and
   decompose it into ≤ 10-line functions with docstrings; behaviour
   must be unchanged.
7. **Structure chooser** — for each task say the structure and why:
   (a) today's queue of patients, (b) has this username been taken?,
   (c) phone book by name, (d) a fixed RGB colour constant.
8. **Step counter** — instrument the §6 pair-scan with a counter and
   print steps for n = 50, 100, 200; confirm the ×4-per-×2 signature.
9. **Test suite** — build `(input, expected)` tables for
   `count_at_least` from §1 (normal, boundary, empty) and print
   PASS/FAIL lines.
10. **Docstring hunt** — find three of your own functions with weak
    docstrings and rewrite them to say *what is returned*, including
    the special cases.

### Challenge Problems

1. **Algorithm duel** — implement "first repeated value" two ways
   (pair-scan vs set); write the step-count table for n = 100…800 for
   both; conclude in one sentence which you would ship.
2. **Mini framework** — write `run_tests(function, cases)` where cases
   is a list of `(input, expected)`; it prints a PASS/FAIL line per
   case and a final summary line. Use it to test two of your functions.
3. **Project extension** — add per-subject support to the final
   project (nested-dict marks), including stats per subject, and a
   boundary test table for the new `clean_row`.

<a id="15"></a>

## 15. Summary and where to go next

- **Computational thinking** = decomposition + abstraction +
  generalisation + algorithm design, applied before coding.
- **Growth classes:** O(1) indexing/dict-lookup, O(n) single loops,
  O(n²) nested loops — Big-O predicts *growth*, not seconds.
- **Structures are decisions:** list for order, tuple for fixed shape,
  set for membership/uniqueness, dict for keyed lookup — and the
  choice *is* the algorithm.
- **Maintainable code** = meaningful names, small docstringed
  functions, no duplication, why-comments, consistent formatting.
- **Testing** = `(input, expected)` tables covering normal, boundary
  and invalid cases — evidence, re-runnable after every change.
- **Debugging** = reproduce → localise → inspect → fix one thing →
  re-test.
- The **final project** tied it together: records list + roll-index
  dict + used-IDs set, validated load, scripted menu, JSON summary,
  boundary-tested validators.

### Where to go next

You have finished the fundamentals. Natural next steps, in rough order:

1. **Pandas & NumPy** — everything in Module 15 reappears as
   `DataFrame`, `filter`, `groupby`, `mean` — but now you know what the
   library does for you.
2. **More Python** — comprehensions, generators, classes in depth,
   decorators — the "advanced" list from Module 1's promises.
3. **Algorithms & data structures** — sorting, searching, stacks and
   queues — Module 16's Big-O becomes a daily tool.
4. **Version control (git)** and **virtual environments** (Notebook 00)
   in real projects.

**Congratulations** — you can now read, write, test and reason about
Python programs. That skill compounds for the rest of your career.